In [0]:
# Day 4: Delta Lake Introduction
#
# Topics:
# - Delta Lake basics
# - ACID transactions
# - Schema enforcement
# - Delta vs Parquet behavior

In [0]:
from pyspark.sql import functions as f

In [0]:
OCT_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"
NOV_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv"

events = (
    spark.read.option("header", True).option("inferSchema", True).csv(OCT_PATH)
    .unionByName(
        spark.read.option("header", True).option("inferSchema", True).csv(NOV_PATH)
    )
)

print(f"Total events: {events.count():,}")
events.printSchema()

# Convert CSV → Delta
DELTA_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/delta/events"

events.write.format("delta").mode("overwrite").save(DELTA_PATH)

In [0]:
# Validation
spark.read.format("delta").load(DELTA_PATH).count()

In [0]:
# Create Delta table using PySpark
# - Managed table approach
events.write.format("delta").mode("overwrite").saveAsTable("events_table")

#Verify
spark.sql("SELECT COUNT(*) FROM events_table").show()

In [0]:
# Create Delta table using SQL
spark.sql("""
    CREATE TABLE IF NOT EXISTS events_delta
    USING DELTA
    AS SELECT * FROM events_table
""")

#Verify 
spark.sql("DESCRIBE DETAIL events_delta").show(truncate=False)

In [0]:
# Test schema enforcement

from pyspark.sql.types import StructType, StructField, StringType

wrong_schema_df = spark.createDataFrame(
    [("a", "b", "c")],
    StructType([
        StructField("x", StringType()),
        StructField("y", StringType()),
        StructField("z", StringType())
    ])
)

try:
    wrong_schema_df.write.format("delta").mode("append").save(DELTA_PATH)
except Exception as e:
    print("Schema enforcement triggered:")
    print(e)


In [0]:
# Handle duplicate inserts
events.write.format("delta").mode("append").save(DELTA_PATH)

spark.read.format("delta").load(DELTA_PATH).count()


In [0]:
# Delta vs Parquet
# Parquet: no ACID, no schema enforcement
# Delta  : ACID transactions, schema enforcement, versioning

PARQUET_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/parquet/events_parquet"

events.write.format("parquet").mode("overwrite").save(PARQUET_PATH)

# validate
spark.read.format("parquet").load(PARQUET_PATH).count()
